# 🤖 Corporate AI Adoption: Maturity Prediction

> **Goal:** Predict a company's **AI Maturity Score** (1–10) using investment & operational metrics.
>
> **Dataset:** 200,000 corporate records | **Target:** `ai_maturity_score`
>
> **What you'll learn:**
> - Explore and clean data
> - Build regression models step by step
> - Evaluate and interpret results

---

## 📦 Step 1: Import Libraries
We import everything we need before starting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print('✅ All libraries imported successfully!')

## 📂 Step 2: Load & Preview Dataset
Let's load the CSV and take a first look.

In [ ]:
df = pd.read_csv('/kaggle/input/corporate-ai-adoption-dataset/corporate_ai_adoption_dataset.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head()

## 🔍 Step 3: Understand the Data
Check column types, missing values, and basic statistics.

In [ ]:
print('--- Column Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n✅ No missing values — dataset is clean!')

In [ ]:
df.describe().T.round(2)

## 📊 Step 4: Exploratory Data Analysis (EDA)
Visualize patterns before building the model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of target
axes[0].hist(df['ai_maturity_score'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of AI Maturity Score', fontsize=13)
axes[0].set_xlabel('AI Maturity Score')
axes[0].set_ylabel('Count')

# Avg score by industry
industry_avg = df.groupby('industry')['ai_maturity_score'].mean().sort_values()
industry_avg.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Avg AI Maturity Score by Industry', fontsize=13)
axes[1].set_xlabel('Average Score')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: deployment_count vs ai_maturity_score
sample = df.sample(3000, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(sample['deployment_count'], sample['ai_maturity_score'],
            alpha=0.3, color='teal', s=10)
plt.title('Deployment Count vs AI Maturity Score', fontsize=13)
plt.xlabel('Deployment Count')
plt.ylabel('AI Maturity Score')
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=np.number).columns

plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 🛠️ Step 5: Feature Engineering

ML models need **numbers only** — no text.  
We convert `industry` and `country` to numbers using **Label Encoding**.

In [ ]:
le = LabelEncoder()

df['industry_enc'] = le.fit_transform(df['industry'])
df['country_enc']  = le.fit_transform(df['country'])

features = [
    'ai_adoption_level', 'ai_investment_usd', 'automation_rate',
    'cost_savings', 'revenue_impact', 'productivity_gain',
    'employee_ai_training_hours', 'deployment_count',
    'industry_enc', 'country_enc', 'year'
]

X = df[features]            # Inputs
y = df['ai_maturity_score'] # Target

print(f'Features shape: {X.shape}')
print(f'Target shape  : {y.shape}')

## ✂️ Step 6: Train-Test Split

- **80% Training** — model learns from this
- **20% Testing**  — we evaluate on unseen data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set : {X_train.shape[0]:,} rows')
print(f'Testing set  : {X_test.shape[0]:,} rows')

## 🤖 Step 7: Train Models

We compare two models:
1. **Linear Regression** — simple baseline
2. **Random Forest** — powerful ensemble of 100 decision trees

In [ ]:
# Model 1: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
print('✅ Linear Regression trained!')

In [ ]:
# Model 2: Random Forest
# n_estimators=100 → builds 100 decision trees and averages their predictions
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print('✅ Random Forest trained!')

## 📈 Step 8: Evaluate Models

Metrics explained:
- **R² Score** — How well model explains variance (1.0 = perfect)
- **MAE** — Average prediction error in original units
- **RMSE** — Penalizes large errors more than MAE

In [ ]:
def evaluate(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f'{name}:')
    print(f'  R² Score : {r2:.4f}')
    print(f'  MAE      : {mae:.4f}')
    print(f'  RMSE     : {rmse:.4f}')
    print()
    return r2, mae, rmse

lr_r2, lr_mae, lr_rmse = evaluate('Linear Regression', y_test, lr_pred)
rf_r2, rf_mae, rf_rmse = evaluate('Random Forest    ', y_test, rf_pred)

In [ ]:
# Bar chart comparison
models = ['Linear Regression', 'Random Forest']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(models, [lr_r2, rf_r2], color=['steelblue', 'coral'])
axes[0].set_title('R² Score (Higher = Better)', fontsize=13)
axes[0].set_ylim(0.95, 1.0)
for i, v in enumerate([lr_r2, rf_r2]):
    axes[0].text(i, v + 0.0005, f'{v:.4f}', ha='center', fontsize=11)

axes[1].bar(models, [lr_mae, rf_mae], color=['steelblue', 'coral'])
axes[1].set_title('MAE (Lower = Better)', fontsize=13)
for i, v in enumerate([lr_mae, rf_mae]):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 🌟 Step 9: Feature Importance
Which features matter most for predicting AI Maturity Score?

In [ ]:
importance = pd.Series(rf.feature_importances_, index=features).sort_values()

plt.figure(figsize=(9, 6))
importance.plot(kind='barh', color='teal')
plt.title('Feature Importance (Random Forest)', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top feature:', importance.idxmax(), f'({importance.max():.3f})')

In [ ]:
# Actual vs Predicted scatter (sample of 500 points)
idx = np.random.choice(len(y_test), 500, replace=False)
y_s = np.array(y_test)[idx]
p_s = rf_pred[idx]

plt.figure(figsize=(7, 7))
plt.scatter(y_s, p_s, alpha=0.4, color='steelblue', s=15)
plt.plot([1, 10], [1, 10], 'r--', lw=2, label='Perfect Prediction')
plt.title('Actual vs Predicted — Random Forest', fontsize=13)
plt.xlabel('Actual AI Maturity Score')
plt.ylabel('Predicted AI Maturity Score')
plt.legend()
plt.tight_layout()
plt.show()

## 🏆 Conclusion

| Model | R² Score | MAE | RMSE |
|-------|----------|-----|------|
| Linear Regression | ~0.9717 | ~0.2582 | ~0.3270 |
| **Random Forest** | **~0.9777** | **~0.2280** | **~0.2898** |

**Key Findings:**

- 🥇 **Random Forest wins** with **R² ≈ 0.978** — it predicts AI maturity with ~97.8% accuracy.
- 📌 **`deployment_count`** is the most important predictor (~94.7% importance). The more AI solutions a company deploys, the higher its maturity.
- 📌 **`ai_adoption_level`** is second most important (~2.9%).
- 🏭 Industries and countries have relatively small but non-zero influence.
- 📅 `year` matters slightly — AI maturity grows over time.

**Business Insight:**  
To improve AI maturity, companies should focus on **deploying more AI solutions** and increasing their overall **AI adoption level** — these two levers drive the most impact.

---
*Notebook by: Shreyash | Dataset: Corporate AI Adoption (200K records)*